# Clustering algorithm: gmm
Visualise the representative grids of the Gaussian mixture model clustering (tied covariance).

> **Note:** This notebook needs a classification data set of many postcode areas (sample set, generated and analysed grids, clustering parameters; see the classification documentation and `pylovo-classify`).

In [ ]:
import warnings

from pylovo.classification.clustering.clustering_algorithms import gmm_tied_clustering
from pylovo.classification.database_communication.database_communication import DatabaseCommunication
from pylovo.database.database_client import DatabaseClient
from pylovo.plotting.classification import plot_radar_graph
from pylovo.plotting.generation.networks import (
    get_network_info_for_plotting,
    plot_simple_grid,
    plot_with_generic_coordinates,
)

warnings.filterwarnings('ignore')

Import grid parameter data and cluster

In [ ]:
dbc = DatabaseClient()  # read-only access (municipal register)

In [ ]:
dc = DatabaseCommunication()
df_parameters_of_grids = dc.get_clustering_parameters_for_classification_version()
choices = dc.get_classification_choices()  # the parameters and numbers of clusters of the classification

In [ ]:
df_parameters_of_grids, representative_networks = gmm_tied_clustering(df_parameters_of_grids=df_parameters_of_grids,
                                                                      list_of_clustering_parameters=choices["list_of_clustering_parameters"],
                                                                      n_clusters=choices["n_clusters_gmm"])

## Radar Plot
Compare the representative grids in terms of the clustering parameters. Which parameter dominates in the representative grid?

In [ ]:
plot_radar_graph(representative_networks=representative_networks, list_of_parameters=choices["list_of_clustering_parameters"])

In [ ]:
# the key figures of every representative grid, one column per cluster
representative_networks.set_index("clusters").drop(columns=["index", "version_id"]).T

## Representative grids
One grid per cluster, from the smallest representative grid (cluster 0, fewest households) to the largest. The loop covers all clusters, whatever `N_CLUSTERS_GMM` is.

In [ ]:
for _, grid in representative_networks.iterrows():
    print(f"Cluster {grid['clusters']}")
    plz, kcid, bcid = get_network_info_for_plotting(grid)
    display(plot_simple_grid(plz, bcid=bcid, kcid=kcid))
    # plot_grid_on_map(plz, bcid=bcid, kcid=kcid)
    # net = read_net_with_grid_generator(plz, kcid, bcid)
    # pp.to_json(net, f"grid{grid['clusters']}.json")
    # pp.to_excel(net, f"grid{grid['clusters']}.xlsx")
    display(plot_with_generic_coordinates(plz, bcid=bcid, kcid=kcid))
    display(dbc.get_municipal_register_for_plz(int(plz)))